In [1]:
import json
import statistics
import math
import numpy as np

In [2]:
INPUT_PATH = "results_ft.json"

with open(INPUT_PATH, "r") as f:
    data = json.load(f)

In [3]:
fa_all, c1_all, c2_all = [], [], []
fa_flag0, c1_flag0, c2_flag0 = [], [], []
fa_flag1, c1_flag1, c2_flag1 = [], [], []

count_flag0 = 0
count_flag1 = 0

In [4]:
for instance in data:
    fa = instance["avg_FA"]
    c1 = instance["avg_C1"]
    c2 = instance["avg_C2"]
    flag = instance["final_Flag"]

    fa_all.append(fa)
    c1_all.append(c1)
    c2_all.append(c2)

    if flag == 0:
        count_flag0 += 1
        fa_flag0.append(fa)
        c1_flag0.append(c1)
        c2_flag0.append(c2)
    elif flag == 1:
        count_flag1 += 1
        fa_flag1.append(fa)
        c1_flag1.append(c1)
        c2_flag1.append(c2)

In [5]:
def compute_stats(values):
    if len(values) == 0:
        return (0, 0, 0, 0)
    
    mean = statistics.mean(values)
    std = statistics.stdev(values) if len(values) > 1 else 0
    ci95 = 1.96 * (std / math.sqrt(len(values))) if len(values) > 1 else 0
    p95 = np.percentile(values, 95)
    return mean, std, ci95, p95

In [6]:
def print_stats(label, stats):
    mean, std, ci95, p95 = stats
    print(f"{label}:")
    print(f"  Mean ± Std       : {mean:.3f} ± {std:.3f}")
    print(f"  95% CI           : ± {ci95:.3f}")
    print(f"  95th Percentile  : {p95:.3f}")

In [7]:
fa_all_stats = compute_stats(fa_all)
c1_all_stats = compute_stats(c1_all)
c2_all_stats = compute_stats(c2_all)

fa_0_stats = compute_stats(fa_flag0)
c1_0_stats = compute_stats(c1_flag0)
c2_0_stats = compute_stats(c2_flag0)

fa_1_stats = compute_stats(fa_flag1)
c1_1_stats = compute_stats(c1_flag1)
c2_1_stats = compute_stats(c2_flag1)

In [8]:
print("=== OVERALL ===")
print(f"Total Samples: {len(data)}")
print_stats("FA", fa_all_stats)
print_stats("C1", c1_all_stats)
print_stats("C2", c2_all_stats)

print("\n=== FLAG = 0 ===")
print(f"Count: {count_flag0}")
print_stats("FA", fa_0_stats)
print_stats("C1", c1_0_stats)
print_stats("C2", c2_0_stats)

print("\n=== FLAG = 1 ===")
print(f"Count: {count_flag1}")
print_stats("FA", fa_1_stats)
print_stats("C1", c1_1_stats)
print_stats("C2", c2_1_stats)

=== OVERALL ===
Total Samples: 179
FA:
  Mean ± Std       : 9.158 ± 1.777
  95% CI           : ± 0.260
  95th Percentile  : 10.000
C1:
  Mean ± Std       : 9.119 ± 1.803
  95% CI           : ± 0.264
  95th Percentile  : 10.000
C2:
  Mean ± Std       : 9.114 ± 1.133
  95% CI           : ± 0.166
  95th Percentile  : 10.000

=== FLAG = 0 ===
Count: 169
FA:
  Mean ± Std       : 9.215 ± 1.787
  95% CI           : ± 0.269
  95th Percentile  : 10.000
C1:
  Mean ± Std       : 9.179 ± 1.814
  95% CI           : ± 0.274
  95th Percentile  : 10.000
C2:
  Mean ± Std       : 9.144 ± 1.148
  95% CI           : ± 0.173
  95th Percentile  : 10.000

=== FLAG = 1 ===
Count: 10
FA:
  Mean ± Std       : 8.200 ± 1.317
  95% CI           : ± 0.816
  95th Percentile  : 10.000
C1:
  Mean ± Std       : 8.100 ± 1.287
  95% CI           : ± 0.797
  95th Percentile  : 10.000
C2:
  Mean ± Std       : 8.600 ± 0.699
  95% CI           : ± 0.433
  95th Percentile  : 9.550


In [9]:
import pandas as pd

df = pd.DataFrame({
    "FA": fa_all,
    "C1": c1_all,
    "C2": c2_all
})

correlation_matrix = df.corr(method="pearson")

print("\n=== Correlation Matrix ===")
print(correlation_matrix)



=== Correlation Matrix ===
          FA        C1        C2
FA  1.000000  0.994213  0.916601
C1  0.994213  1.000000  0.919625
C2  0.916601  0.919625  1.000000


In [10]:
from scipy import stats

t_stat_fa, p_value_fa = stats.ttest_ind(fa_flag0, fa_flag1, equal_var=False)
t_stat_c1, p_value_c1 = stats.ttest_ind(c1_flag0, c1_flag1, equal_var=False)
t_stat_c2, p_value_c2 = stats.ttest_ind(c2_flag0, c2_flag1, equal_var=False)

print("\n=== T-Test (FA: Flag 0 vs Flag 1) ===")
print("t-statistic:", t_stat_fa)
print("p-value:", p_value_fa)
if p_value_fa < 0.05:
    print("✅ Statistically significant difference in FA between Flag 0 and Flag 1")
else:
    print("❌ No statistically significant difference in FA between Flag 0 and Flag 1")

print("\n=== T-Test (C1: Flag 0 vs Flag 1) ===")
print("t-statistic:", t_stat_c1)
print("p-value:", p_value_c1)
if p_value_c1 < 0.05:
    print("✅ Statistically significant difference in C1 between Flag 0 and Flag 1")
else:
    print("❌ No statistically significant difference in C1 between Flag 0 and Flag 1")

print("\n=== T-Test (C2: Flag 0 vs Flag 1) ===")
print("t-statistic:", t_stat_c2)
print("p-value:", p_value_c2)
if p_value_c2 < 0.05:
    print("✅ Statistically significant difference in C2 between Flag 0 and Flag 1")
else:
    print("❌ No statistically significant difference in C2 between Flag 0 and Flag 1")



=== T-Test (FA: Flag 0 vs Flag 1) ===
t-statistic: 2.3149996337665306
p-value: 0.040813448380518175
✅ Statistically significant difference in FA between Flag 0 and Flag 1

=== T-Test (C1: Flag 0 vs Flag 1) ===
t-statistic: 2.509560299527858
p-value: 0.02861302302714854
✅ Statistically significant difference in C1 between Flag 0 and Flag 1

=== T-Test (C2: Flag 0 vs Flag 1) ===
t-statistic: 2.2847153398765494
p-value: 0.041181902171635636
✅ Statistically significant difference in C2 between Flag 0 and Flag 1


In [11]:
data.sort(key=lambda x: x["avg_FA"], reverse=True)
top_50_ft = data[:50]

In [12]:
top_50_ft[0]

{'CNR': 'KLHC010074352013',
 'case_details': 'Applicant applied for regular-bail. The age of the applicant/s is/are [27]. The health condition of the applicant is none. Past criminal records for the applicant do not exist. The relevant statutes are: 376 IPC, 156 CrPC, 34 IPC. The applicant is in custody for 15 days. The petitioner is the first accused in Crime No. 1290 of 2013 of Kadavanthra Police Station, who is alleged to have committed offences punishable under Section 376 read with Section 34 of Indian Penal Code. The allegation is that the petitioner herein, who is the first accused, was staying with the second accused who had let him to occupy a hostel run by her. The allegation is that the first accused with the connivance of the second accused began to induce the victim to develop affinity towards the first accused. Though the victim was reluctant to do so and she had expressed her disinclination for that course, the second accused pursued her attempt. It is alleged that when 

In [13]:
top_50_ft_dict = {item["CNR"]: item for item in top_50_ft}

In [14]:
with open("results.json", "r", encoding="utf-8") as f:
    data_baseline = json.load(f)

In [15]:
data_baseline[0]

{'CNR': 'KLHC010042122018',
 'case_details': 'Applicant applied for anticipatory-bail. The age of the applicant/s is/are [37]. The health condition of the applicant is none. Past criminal records for the applicant do not exist. The relevant statutes are: 449 IPC, 302 IPC, 392 IPC. The applicant is in custody for 177 days. The accused is charged with the murder of a 47-year-old housewife inside her house, suspected to be a murder for gain as valuables were stolen. The accused allegedly trespassed into her house, attempted to sexually abuse her, attacked her with a broken ceremonial lamp and a stick, and smothered her causing her death. The accused is also alleged to have taken away gold ornaments, cash, a mobile phone, and the weapons used for the commission of the offence.\nArguments supporting the bail application are The investigation having been completed, there is no justification in denying him bail. The incident was in 2015, and the present investigating officer was in charge for

In [16]:
expert_eval_samples = []
for item in data_baseline:
    if item["CNR"] in top_50_ft_dict.keys():
        sample = {
            "CNR": item["CNR"],
            "case_details": item["case_details"],
            "outcome": item["outcome"],
            "statutes_info": item["statutes_info"],
            "reason_court": top_50_ft_dict[item["CNR"]]["reason_court"],
            "reason_ft": top_50_ft_dict[item["CNR"]]["reason_machine"],
            "reason_baseline": item["reason_machine"],
            "baseline_results":{
                "avg_FA": item["avg_FA"],
                "avg_C1": item["avg_C1"],
                "avg_C2": item["avg_C2"],
                "std_FA": item["std_FA"],
                "std_C1": item["std_C1"],
                "std_C2": item["std_C2"],
                "final_Flag": item["final_Flag"]
            },
            "ft_results":{
                "avg_FA": top_50_ft_dict[item["CNR"]]["avg_FA"],
                "avg_C1": top_50_ft_dict[item["CNR"]]["avg_C1"],
                "avg_C2": top_50_ft_dict[item["CNR"]]["avg_C2"],
                "std_FA": top_50_ft_dict[item["CNR"]]["std_FA"],
                "std_C1": top_50_ft_dict[item["CNR"]]["std_C1"],
                "std_C2": top_50_ft_dict[item["CNR"]]["std_C2"],
                "final_Flag": top_50_ft_dict[item["CNR"]]["final_Flag"]
            }
        }
        expert_eval_samples.append(sample)
        

In [17]:
expert_eval_samples[0]

{'CNR': 'KLHC010074352013',
 'case_details': 'Applicant applied for regular-bail. The age of the applicant/s is/are [27]. The health condition of the applicant is none. Past criminal records for the applicant do not exist. The relevant statutes are: 376 IPC, 156 CrPC, 34 IPC. The applicant is in custody for 15 days. The petitioner is the first accused in Crime No. 1290 of 2013 of Kadavanthra Police Station, who is alleged to have committed offences punishable under Section 376 read with Section 34 of Indian Penal Code. The allegation is that the petitioner herein, who is the first accused, was staying with the second accused who had let him to occupy a hostel run by her. The allegation is that the first accused with the connivance of the second accused began to induce the victim to develop affinity towards the first accused. Though the victim was reluctant to do so and she had expressed her disinclination for that course, the second accused pursued her attempt. It is alleged that when 

In [ ]:
# with open("../Expert eval/expert_eval_samples.json", "w", encoding="utf-8") as f:
#     json.dump(expert_eval_samples, f, indent=4, ensure_ascii=False)